# LLaVA-Interleave (0.5B) — Experiment B, Zero-Shot

Same model and technical setup as `LLaVA_ExperimentA_ZeroShot.ipynb`, including `min_new_tokens=18` — kept deliberately for consistency with every other LLaVA experiment in this project, and confirmed by diagnostic testing not to change the underlying findings even though it does affect which specific degenerate pattern appears (see Experiment A's findings doc, Section 8).

Same zero-shot system prompt, with the same two-line YOLO block used in every other Experiment B in this project. Test image + YOLO count only — still no support images.

## 1. Mount Drive & confirm input data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
from zipfile import ZipFile

zip_path = "/content/drive/MyDrive/CrowdDensityResearch/Phase5_Input.zip"

if not os.path.exists("/content/Phase5_Input"):
    with ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall("/content/")

print("Input ready:", os.path.exists("/content/Phase5_Input"))

Input ready: True


## 2. Back up any existing zero-shot B results before overwriting

In [ ]:
import os, shutil

RESULT_DIR = "/content/drive/MyDrive/CrowdDensityResearch/VLM_Results"
os.makedirs(RESULT_DIR, exist_ok=True)

for fname in ["llava_experiment_B_zeroshot.csv", "llava_experiment_B_zeroshot_evaluated.csv"]:
    src = os.path.join(RESULT_DIR, fname)
    if os.path.exists(src):
        backup = os.path.join(RESULT_DIR, fname.replace(".csv", "_backup.csv"))
        if not os.path.exists(backup):
            shutil.copy2(src, backup)
            print(f"Backed up {fname} -> {os.path.basename(backup)}")
        else:
            print(f"Backup already exists for {fname}, skipping.")
    else:
        print(f"No existing {fname} found — nothing to back up.")

No existing llava_experiment_B_zeroshot.csv found — nothing to back up.
No existing llava_experiment_B_zeroshot_evaluated.csv found — nothing to back up.


## 3. Install packages

In [ ]:
!pip install -q -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 94.9 MB/s eta 0:00:00


## 4. Load the model (identical to every other LLaVA experiment)

In [ ]:
import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration

MODEL_NAME = "llava-hf/llava-interleave-qwen-0.5b-hf"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_NAME)

print("Loading model...")
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()

print("✅ Model loaded successfully!")

CUDA available: True
GPU: Tesla T4
Loading processor...


processor_config.json:   0%|          | 0.00/169 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/699 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/393 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.25k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.60k [00:00<?, ?B/s]

... (5 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/715 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/126 [00:00<?, ?B/s]

✅ Model loaded successfully!


## 5. Helpers

In [ ]:
from PIL import Image

def load_image(path, max_size=512):
    """Load an image and resize it to reduce memory usage."""
    img = Image.open(path).convert("RGB")
    img.thumbnail((max_size, max_size))
    return img

DENSITY_TEXT = {
    0: "Low",
    1: "Medium",
    2: "High"
}

## 6. Zero-shot Experiment A's exact prompt (reference only — not re-run here)
Kept verbatim so Section 8's diff check can prove B differs from A only where intended.

In [ ]:
def build_messages_zeroshot_A(test_image):
    """LLaVA conversation for one test image — zero-shot Experiment A (reference only)."""
    conversation = []
    images = []

    conversation.append({
        "role": "system",
        "content": [{
            "type": "text",
            "text": """You are an expert in crowd counting and crowd density estimation.

You will receive one image.

Predict:

1. The EXACT crowd count.
2. The crowd density.

Density labels:
0 = Low
1 = Medium
2 = High

Return ONLY:

Count: <integer>
Density: <0/1/2>

Do not explain.
Do not estimate.
Do not output ranges.
Return exactly one integer count.
"""
        }]
    })

    conversation.append({
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": """TEST IMAGE

Count every visible person exactly once.

Return ONLY

Count: <integer>

Density: <0/1/2>
"""}
        ]
    })
    images.append(load_image(test_image))

    return conversation, images

## 7. Zero-shot Experiment B — A's template + one YOLO block

In [ ]:
def build_messages_zeroshot_B(test_image, yolo_count):
    """
    LLaVA conversation for one test image — zero-shot Experiment B.
    Identical to build_messages_zeroshot_A() above except for the YOLO block marked below.
    """
    conversation = []
    images = []

    conversation.append({
        "role": "system",
        "content": [{
            "type": "text",
            "text": """You are an expert in crowd counting and crowd density estimation.

You will receive one image.

Predict:

1. The EXACT crowd count.
2. The crowd density.

Density labels:
0 = Low
1 = Medium
2 = High

Return ONLY:

Count: <integer>
Density: <0/1/2>

Do not explain.
Do not estimate.
Do not output ranges.
Return exactly one integer count.
"""
        }]
    })

    # Only this final turn differs from build_messages_zeroshot_A() above —
    # the two lines giving the YOLO count, same wording as every other Experiment B.
    conversation.append({
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": f"""TEST IMAGE

YOLO predicted count: {int(yolo_count)}

Use the YOLO predicted count as additional information, but inspect the image yourself.

Count every visible person exactly once.

Return ONLY

Count: <integer>

Density: <0/1/2>
"""}
        ]
    })
    images.append(load_image(test_image))

    return conversation, images

## 8. Inference, parsing, and verification

In [ ]:
import gc
import re
import torch


def run_llava(conversation, images):
    prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)

    inputs = processor(images=images, text=prompt, return_tensors="pt")
    inputs = {k: v.to(model.device, torch.float16) if v.dtype.is_floating_point else v.to(model.device)
              for k, v in inputs.items()}

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs, max_new_tokens=40, min_new_tokens=18, do_sample=False
        )

    trimmed_ids = generated_ids[:, inputs["input_ids"].shape[1]:]
    response = processor.batch_decode(
        trimmed_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]

    del inputs, generated_ids, trimmed_ids
    gc.collect()
    torch.cuda.empty_cache()

    return response.strip()


def parse_response(response):
    """Rejects degenerate long digit-runs as unparseable rather than
    taking them literally (same guard used for the final LLaVA runs)."""
    response = str(response)

    count = None
    m = re.search(r"Count\s*[:=]\s*(\d+)", response, re.IGNORECASE)
    if m:
        digits = m.group(1)
        if len(digits) <= 5:
            count = int(digits)

    density = None
    m = re.search(r"Density\s*[:=]\s*([012])", response, re.IGNORECASE)
    if m:
        density = int(m.group(1))
    else:
        r = response.lower()
        if "low" in r:
            density = 0
        elif "medium" in r:
            density = 1
        elif "high" in r:
            density = 2

    return count, density

## 9. Load data and merge YOLO predictions
No `support_mapping.csv` — just `test.csv` merged directly with `yolo_predictions.csv`.

In [ ]:
import pandas as pd

ROOT = "/content/Phase5_Input"
TEST_DIR = f"{ROOT}/test/images"

test_df = pd.read_csv(f"{ROOT}/test.csv")
yolo_df = pd.read_csv(f"{ROOT}/yolo_predictions.csv")

test_B = test_df.merge(yolo_df, left_on="filename", right_on="image_id", how="left")

print("Total rows:", len(test_B))
print("Missing YOLO counts:", test_B["predicted_count"].isna().sum())

n_before = len(test_B)
test_B = test_B.dropna(subset=["predicted_count"]).reset_index(drop=True)
print(f"Dropped {n_before - len(test_B)} rows with no YOLO match; {len(test_B)} usable rows remain.")

Total rows: 1600
Missing YOLO counts: 0
Dropped 0 rows with no YOLO match; 1600 usable rows remain.


## 10. Verify B differs from A by exactly the YOLO block

In [ ]:
_sample = test_B.iloc[0]
_test_image = os.path.join(TEST_DIR, f"{int(_sample['filename']):04d}.jpg")

_conv_A, _ = build_messages_zeroshot_A(_test_image)
_conv_B, _ = build_messages_zeroshot_B(_test_image, int(_sample["predicted_count"]))

def _texts(conv):
    out = []
    for m in conv:
        for c in m["content"]:
            if c.get("type") == "text":
                out.append(c["text"])
    return out

_texts_A, _texts_B = _texts(_conv_A), _texts(_conv_B)
print("System prompt identical:", _texts_A[0] == _texts_B[0])
print()
print("--- Test-image turn, A ---")
print(_texts_A[-1])
print("--- Test-image turn, B ---")
print(_texts_B[-1])

System prompt identical: True

--- Test-image turn, A ---
TEST IMAGE

Count every visible person exactly once.

Return ONLY

Count: <integer>

Density: <0/1/2>

--- Test-image turn, B ---
TEST IMAGE

YOLO predicted count: 32

Use the YOLO predicted count as additional information, but inspect the image yourself.

Count every visible person exactly once.

Return ONLY

Count: <integer>

Density: <0/1/2>



## 11. Smoke test — check 3 samples before running the full loop

In [ ]:
N_SMOKE = 3

for idx in range(N_SMOKE):
    sample_i = test_B.iloc[idx]
    test_image_i = os.path.join(TEST_DIR, f"{int(sample_i['filename']):04d}.jpg")

    conversation_i, images_i = build_messages_zeroshot_B(test_image_i, int(sample_i["predicted_count"]))
    response_i = run_llava(conversation_i, images_i)
    parsed_i = parse_response(response_i)

    print(f"--- Sample {idx} (filename={int(sample_i['filename'])}, true_count={int(sample_i['total_count'])}, yolo={int(sample_i['predicted_count'])}) ---")
    print("Response:", repr(response_i))
    print("Parsed  :", parsed_i)
    if parsed_i[0] is None or parsed_i[1] is None:
        print("⚠️  Incomplete parse.")
    print()

--- Sample 0 (filename=2, true_count=43, yolo=32) ---
Response: 'Count: 32\nDensity: 0.25/1000'
Parsed  : (32, 0)

--- Sample 1 (filename=4, true_count=59, yolo=25) ---
Response: 'Count: 25\nDensity: 0.25/1000'
Parsed  : (25, 0)

--- Sample 2 (filename=5, true_count=463, yolo=1) ---
Response: 'Count: 1000\nDensity: 0.1/1000'
Parsed  : (1000, 0)



## 12. Full run — all usable test images, saved continuously to Drive

In [ ]:
from tqdm import tqdm
import os
import gc
import time
import torch

OUT_PATH = os.path.join(RESULT_DIR, "llava_experiment_B_zeroshot.csv")

if "results_B_zs" not in globals():
    if os.path.exists(OUT_PATH):
        prior = pd.read_csv(OUT_PATH)
        results_B_zs = prior.to_dict("records")
        for r in results_B_zs:
            r["test_filename"] = int(r["test_filename"])
        print(f"Resumed {len(results_B_zs)} results from a previous session's CSV: {OUT_PATH}")
    else:
        results_B_zs = []

done_filenames = {r["test_filename"] for r in results_B_zs}
remaining = test_B[~test_B["filename"].astype(int).isin(done_filenames)]
print(f"Already done: {len(done_filenames)} | Remaining: {len(remaining)}")

MAX_RETRIES_PER_IMAGE = 2

for _, sample in tqdm(remaining.iterrows(), total=len(remaining)):

    test_image = os.path.join(TEST_DIR, f"{int(sample['filename']):04d}.jpg")

    conversation, images = build_messages_zeroshot_B(test_image, int(sample["predicted_count"]))

    response = None
    for attempt in range(1, MAX_RETRIES_PER_IMAGE + 1):
        try:
            response = run_llava(conversation, images)
            break
        except RuntimeError as e:
            if "out of memory" not in str(e).lower():
                raise
            print(f"\n[OOM] filename={int(sample['filename'])} "
                  f"attempt {attempt}/{MAX_RETRIES_PER_IMAGE} — clearing cache and retrying...")
            gc.collect()
            torch.cuda.empty_cache()
            time.sleep(2)

    if response is None:
        pred_count, pred_density = None, None
        response = "ERROR: CUDA out of memory after retries"
    else:
        pred_count, pred_density = parse_response(response)

    results_B_zs.append({
        "test_filename": int(sample["filename"]),
        "yolo_predicted_count": int(sample["predicted_count"]),
        "predicted_count": pred_count,
        "predicted_density": pred_density,
        "raw_response": response
    })

    pd.DataFrame(results_B_zs).to_csv(OUT_PATH, index=False)

failed = sum(1 for r in results_B_zs if r["predicted_count"] is None)
print("Finished:", len(results_B_zs), f"({failed} unparseable)")
print("Saved to:", OUT_PATH)

Already done: 0 | Remaining: 1600


100%|██████████| 1600/1600 [26:13<00:00,  1.02it/s]

Finished: 1600 (1 unparseable)
Saved to: /content/drive/MyDrive/CrowdDensityResearch/VLM_Results/llava_experiment_B_zeroshot.csv


## 13. Evaluate — counting metrics

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

llava_B_zs = pd.DataFrame(results_B_zs)

llava_B_zs_eval = llava_B_zs.merge(
    test_df[["filename", "total_count", "crowd_density"]],
    left_on="test_filename",
    right_on="filename",
    how="left"
)

print("Missing ground truth after merge:", llava_B_zs_eval["total_count"].isna().sum())

n_before = len(llava_B_zs_eval)
count_valid = llava_B_zs_eval.dropna(subset=["predicted_count"]).copy()
n_dropped = n_before - len(count_valid)
print(f"Dropped {n_dropped}/{n_before} rows with no parseable count ({100*n_dropped/n_before:.1f}%)")

y_true_count = count_valid["total_count"]
y_pred_count = count_valid["predicted_count"]

mae = mean_absolute_error(y_true_count, y_pred_count)
rmse = np.sqrt(mean_squared_error(y_true_count, y_pred_count))

nonzero = count_valid["total_count"] > 0
mape = np.mean(np.abs(
    (count_valid.loc[nonzero, "total_count"] - count_valid.loc[nonzero, "predicted_count"])
    / count_valid.loc[nonzero, "total_count"]
)) * 100

print("LLaVA Experiment B (Zero-Shot) — Counting")
print("===========================================")
print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"MAPE : {mape:.2f}%")
print(f"(computed on {len(count_valid)}/{n_before} images with a parseable count)")

Missing ground truth after merge: 0
Dropped 1/1600 rows with no parseable count (0.1%)
LLaVA Experiment B (Zero-Shot) — Counting
MAE  : 233.86
RMSE : 679.08
MAPE : 53.29%
(computed on 1599/1600 images with a parseable count)


## 14. Evaluate — density classification

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

density_valid = llava_B_zs_eval.dropna(subset=["predicted_density"]).copy()
n_density_dropped = len(llava_B_zs_eval) - len(density_valid)
print(f"Dropped {n_density_dropped}/{len(llava_B_zs_eval)} rows with no parseable density ({100*n_density_dropped/len(llava_B_zs_eval):.1f}%)")

y_true_density = density_valid["crowd_density"]
y_pred_density = density_valid["predicted_density"]

accuracy = accuracy_score(y_true_density, y_pred_density)
precision = precision_score(y_true_density, y_pred_density, average="weighted", zero_division=0)
recall = recall_score(y_true_density, y_pred_density, average="weighted", zero_division=0)
f1 = f1_score(y_true_density, y_pred_density, average="weighted", zero_division=0)

print("LLaVA Experiment B (Zero-Shot) — Density")
print("==========================================")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

cm = confusion_matrix(y_true_density, y_pred_density, labels=[0, 1, 2])
print()
print("Confusion matrix (rows=GT, cols=pred):")
print(cm)

Dropped 0/1600 rows with no parseable density (0.0%)
LLaVA Experiment B (Zero-Shot) — Density
Accuracy : 0.1456
Precision: 0.0555
Recall   : 0.1456
F1-score : 0.0465

Confusion matrix (rows=GT, cols=pred):
[[ 222    0    0]
 [ 304   11    0]
 [1013   50    0]]


## 15. Save evaluated results to Drive

In [ ]:
EVAL_PATH = os.path.join(RESULT_DIR, "llava_experiment_B_zeroshot_evaluated.csv")
llava_B_zs_eval.to_csv(EVAL_PATH, index=False)

print("Saved:", len(llava_B_zs_eval), "rows")
print("Path:", EVAL_PATH)

Saved: 1600 rows
Path: /content/drive/MyDrive/CrowdDensityResearch/VLM_Results/llava_experiment_B_zeroshot_evaluated.csv
